> 在线讲义：[第 5 章 · 电气与计算架构：大脑、小脑、神经和血管](https://wikieden.github.io/robot-architecture/lessons/05-electrical-compute.html) · 本书《机器人软硬件架构》

# 第 5 章 · 电气与计算架构：大脑、小脑、神经和血管

配套讲义：仓库里 `robot-system-architecture/notes/lessons/05-electrical-compute.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（EtherCAT 300 字节、30 个从站、电池 48 V × 10 Ah 是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第 2 步 一帧 CAN | 位数 ÷ 速率 | `frame_time_s(130, 1e6)` | 0.13 ms（110 位是 0.11 ms） |
| 第 2 步 一问一答 | 2 帧 | `can_cycle_s(1)` | 0.26 ms |
| 第 2 步 6 个关节一路 CAN | $n \times 2 \times$ 帧时间 | `can_cycle_s(6)` | 1.56 ms → 最高 641 Hz |
| 第 2 步 EtherCAT | $8 \times$ 字节 ÷ 100 Mbps + $n \times t_{从站}$ | `ecat_cycle_s(300, 30)` | 24 + 30 = 54 µs → 约 18.5 kHz |
| 数学原理 电池能量 | Wh = V × Ah | `battery_wh(48, 10)` | 480 Wh |

EtherCAT 报文 300 字节、30 个从站、电池 48 V × 10 Ah 是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `frame_time_s`**：一帧在总线上要多久：位数 ÷ 速率。

In [ ]:
def frame_time_s(bits: float, rate_bps: float) -> float:
    """一帧在总线上要多久：位数 ÷ 速率。"""
    return bits / rate_bps

**函数 `can_cycle_s`**：CAN 一问一答一圈：T ≈ n × 2 × (bits/帧 ÷ 速率)。

In [ ]:
def can_cycle_s(n_joints: int, bits_per_frame: float = 130, rate_bps: float = 1e6) -> float:
    """CAN 一问一答一圈：T ≈ n × 2 × (bits/帧 ÷ 速率)。"""
    return n_joints * 2 * frame_time_s(bits_per_frame, rate_bps)

**函数 `ecat_cycle_s`**：EtherCAT 一圈：T ≈ 8 × 字节数 ÷ 100 Mbps + n × 每个从站的延迟。

In [ ]:
def ecat_cycle_s(frame_bytes: float, n_slaves: int, t_slave_s: float = 1e-6, rate_bps: float = 100e6) -> float:
    """EtherCAT 一圈：T ≈ 8 × 字节数 ÷ 100 Mbps + n × 每个从站的延迟。"""
    return 8 * frame_bytes / rate_bps + n_slaves * t_slave_s

**函数 `f_max`**：最高控制频率 f_max = 1 / T_cycle。

In [ ]:
def f_max(t_cycle_s: float) -> float:
    """最高控制频率 f_max = 1 / T_cycle。"""
    return 1.0 / t_cycle_s

**函数 `battery_wh`**：电池能量 Wh = V × Ah。

In [ ]:
def battery_wh(volt: float, amp_hour: float) -> float:
    """电池能量 Wh = V × Ah。"""
    return volt * amp_hour

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步：CAN 一帧多久（标准帧 110–130 位，1 Mbps）")
for bits in (110, 130):
    print(f"  {bits} 位 → {frame_time_s(bits, 1e6) * 1e3:.2f} ms")
print(f"  每个关节发 1 帧 + 回 1 帧 ≈ {can_cycle_s(1) * 1e3:.2f} ms")

In [ ]:
print("\n第二步：6 个关节挂一路 CAN，一圈要多久")
t = can_cycle_s(6)
print(f"  T ≈ 6 × 2 × 0.13 ms = {t * 1e3:.2f} ms → 最高 {f_max(t):.0f} Hz，跑不到 1 kHz")

In [ ]:
print("\n第二步：EtherCAT 一帧穿过所有从站（编的：300 字节、30 个关节、每站 1 µs）")
print(f"  传输 8 × 300 ÷ 100 Mbps = {8 * 300 / 100e6 * 1e6:.0f} µs")
t = ecat_cycle_s(300, 30)
print(f"  加上 30 个从站 → 一圈 {t * 1e6:.0f} µs → 最高 {f_max(t):.0f} Hz")

In [ ]:
print("\n第三步：电池能量 Wh = V × Ah（编的：48 V × 10 Ah）")
print(f"  {battery_wh(48, 10):.0f} Wh")

## 3. 代码里才看得到的

- 单位最容易错：速率是"位每秒"，报文长度是"字节"，所以 EtherCAT 那里要乘 8。
- 这是理想估算：真实 CAN 还有仲裁、位填充、帧间隔，EtherCAT 还有主站软件的处理时间，实际一圈更长。

## 4. 自己改一改

- 把 `bits_per_frame` 改成 110，看 6 个关节一圈变成多少、能不能到 1 kHz。
- 把 EtherCAT 从站数加到 60，看一圈还剩多少余量。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 大脑算得多但会抖，小脑算得少但准时，关节板只管电流不出错。
> CAN 挨个点名，EtherCAT 一帧走全场；急停走硬线，不走软件。